# Notebook 05
* Author: Melissa Ormeño
* Date: May 2024


Aims:
* Obtiene las mutaciones para enfermedades raras desde orphanet

In [22]:
#Orphanet data
import pandas as pd
import xml.etree.ElementTree as ET

# Cargar y parsear el archivo XML
tree = ET.parse('en_product6 (4).xml')
root = tree.getroot()

# Definir una función para extraer información de cada desorden
def parse_disorder(disorder):
    disorder_data = {}
    disorder_data['DisorderID'] = disorder.attrib.get('id')
    disorder_data['OrphaCode'] = disorder.find('OrphaCode').text if disorder.find('OrphaCode') is not None else None
    disorder_data['ExpertLink'] = disorder.find('ExpertLink').text if disorder.find('ExpertLink') is not None else None
    disorder_data['Name'] = disorder.find('Name').text if disorder.find('Name') is not None else None
    disorder_data['DisorderType'] = disorder.find('DisorderType/Name').text if disorder.find('DisorderType/Name') is not None else None
    disorder_data['DisorderGroup'] = disorder.find('DisorderGroup/Name').text if disorder.find('DisorderGroup/Name') is not None else None

    # Extraer información genética
    gene_info = disorder.find('DisorderGeneAssociationList/DisorderGeneAssociation/Gene')
    if gene_info is not None:
        disorder_data['GeneName'] = gene_info.find('Name').text if gene_info.find('Name') is not None else None
        disorder_data['GeneSymbol'] = gene_info.find('Symbol').text if gene_info.find('Symbol') is not None else None
        disorder_data['GeneLocus'] = gene_info.find('LocusList/Locus/GeneLocus').text if gene_info.find('LocusList/Locus/GeneLocus') is not None else None
    else:
        disorder_data['GeneName'] = None
        disorder_data['GeneSymbol'] = None
        disorder_data['GeneLocus'] = None

    return disorder_data

# Extraer todos los desórdenes en una lista
disorders = []
for disorder in root.find('DisorderList'):
    disorder_data = parse_disorder(disorder)
    disorders.append(disorder_data)

# Convertir la lista de desórdenes en un DataFrame
df = pd.DataFrame(disorders)

# Mostrar el DataFrame
df


,DisorderID,OrphaCode,ExpertLink,Name,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus
0,17601,166024,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Multiple epiphyseal dysplasia-macrocephaly-fac...,Disease,Disorder,kinesin family member 7,KIF7,15q26.1
1,5,93,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Aspartylglucosaminuria,Disease,Disorder,aspartylglucosaminidase,AGA,4q34.3
2,17604,166035,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Brachydactyly-short stature-retinitis pigmento...,Malformation syndrome,Disorder,CWC27 spliceosome associated cyclophilin,CWC27,5q12.3
3,6,585,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Multiple sulfatase deficiency,Disease,Disorder,sulfatase modifying factor 1,SUMF1,3p26.1
4,7,118,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Beta-mannosidosis,Disease,Disorder,mannosidase beta,MANBA,4q24
...,...,...,...,...,...,...,...,...,...
3956,31418,619948,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Early-onset autoimmunity-autoinflammation-immu...,Disease,Disorder,suppressor of cytokine signaling 1,SOCS1,16p13.13
3957,31423,619979,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Developmental delay-immunodeficiency-leukoence...,Disease,Disorder,NFE2 like bZIP transcription factor 2,NFE2L2,2q31.2
3958,31422,619972,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,CADINS disease,Disease,Disorder,caspase recruitment domain family member 11,CARD11,7p22.2
3959,31415,619363,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,NOCARH syndrome,Disease,Disorder,cell division cycle 42,CDC42,1p36.12


In [23]:
df['GeneSymbol'].nunique()

2598

In [24]:
#Data de genes TKL
# Datos con la lista de genes quinasa a trabajar 
#interpro_code = 'IPR000333'
family_name = 'STKR'
df_tkl = pd.read_csv('../data/Cancer/df_icgc_{0}.csv'.format(family_name))

C:\Users\melif\AppData\Local\Temp\ipykernel_33384\2466571243.py:5: DtypeWarning: Columns (26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tkl = pd.read_csv('../data/Cancer/df_icgc_{0}.csv'.format(family_name))


In [25]:
lista_tkl_genes = list(df_tkl['Gen_names'].unique().tolist())
print(len(lista_tkl_genes))
print(sorted(lista_tkl_genes))

12
['ACVR1', 'ACVR1B', 'ACVR1C', 'ACVR2A', 'ACVR2B', 'ACVRL1', 'AMHR2', 'BMPR1A', 'BMPR1B', 'BMPR2', 'TGFBR1', 'TGFBR2']


In [26]:
df[df['GeneSymbol'].str.contains('ACVR1')]


,DisorderID,OrphaCode,ExpertLink,Name,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus
108,205,337,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Fibrodysplasia ossificans progressiva,Disease,Disorder,activin A receptor type 1,ACVR1,2q24.1


In [27]:
df_orphanet = df[df['GeneSymbol'].isin(lista_tkl_genes)]

In [28]:
df_orphanet['GeneSymbol'].unique()

array(['ACVR1', 'ACVRL1', 'BMPR1A', 'BMPR1B', 'BMPR2', 'TGFBR1'],
      dtype=object)

In [29]:
df_orphanet['Name'].unique()

array(['Fibrodysplasia ossificans progressiva',
       'Hereditary hemorrhagic telangiectasia',
       'Hereditary mixed polyposis syndrome',
       'Acromesomelic dysplasia, Grebe type',
       'Fibular aplasia-complex brachydactyly syndrome',
       'Drug- or toxin-induced pulmonary arterial hypertension',
       'Generalized juvenile polyposis/juvenile polyposis coli',
       'Marfan syndrome type 2', 'Familial colorectal cancer Type X',
       'Pulmonary venoocclusive disease',
       'Multiple self-healing squamous epithelioma',
       'Brachydactyly type A1', 'Brachydactyly type C'], dtype=object)

In [30]:
df_orphanet

,DisorderID,OrphaCode,ExpertLink,Name,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus
108,205,337,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Fibrodysplasia ossificans progressiva,Disease,Disorder,activin A receptor type 1,ACVR1,2q24.1
126,236,774,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Hereditary hemorrhagic telangiectasia,Disease,Disorder,activin A receptor like type 1,ACVRL1,12q13.13
634,17145,157794,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Hereditary mixed polyposis syndrome,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2
735,2010,2098,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,"Acromesomelic dysplasia, Grebe type",Malformation syndrome,Disorder,bone morphogenetic protein receptor type 1B,BMPR1B,4q22.3
1020,2413,2639,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Fibular aplasia-complex brachydactyly syndrome,Malformation syndrome,Disorder,bone morphogenetic protein receptor type 1B,BMPR1B,4q22.3
1108,20329,275786,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Drug- or toxin-induced pulmonary arterial hype...,Clinical group,Group of disorders,bone morphogenetic protein receptor type 2,BMPR2,2q33.1-q33.2
1557,21951,329971,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Generalized juvenile polyposis/juvenile polypo...,Clinical subtype,Subtype of disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2
1873,20629,284973,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Marfan syndrome type 2,Clinical subtype,Subtype of disorder,transforming growth factor beta receptor 1,TGFBR1,9q22.33
2327,23409,440437,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Familial colorectal cancer Type X,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2
2568,9801,31837,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Pulmonary venoocclusive disease,Disease,Disorder,bone morphogenetic protein receptor type 2,BMPR2,2q33.1-q33.2


In [31]:
df_orphanet['Name'].nunique()

13

In [32]:
tgfbr1_disease=df_orphanet[df_orphanet['GeneSymbol']=='ACVR2B']
print(tgfbr1_disease['Name'].unique())
tgfbr1_disease

[]


,DisorderID,OrphaCode,ExpertLink,Name,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus


In [12]:
# Data de decipher ya esta filtrada or los genes a tranajar, este filtro se realiza en el notebook 05.Decipher.ipynb
#interpro_code = 'IPR000333'
family_name = 'STKR'
df_decipher = pd.read_csv('../data/mutaciones/decipher_mutations_{0}'.format(family_name))
df_decipher.head(2)

,gene symbol,gene mim,disease name,disease mim,confidence category,allelic requirement,mutation consequence,phenotypes,organ specificity list,pmids,panel,prev symbols,hgnc id,gene disease pair entry date,cross cutting modifier,mutation consequence flag,confidence value flag,comments,variant consequence,disease ontology
0,ACVR2B,602730,HETEROTAXY SYNDROME,207574,limited,monoallelic_autosomal,altered gene product structure,NaN,GI tract;Heart/Cardiovasculature/Lymphatic,NaN,DD,NaN,174,2015-07-22 16:14:34,NaN,NaN,Requires clinical review,NaN,missense_variant;inframe_deletion;inframe_inse...,NaN
1,TGFBR1,190181,LOEYS-DIETZ SYNDROME,609192,definitive,monoallelic_autosomal,altered gene product structure,HP:0000278;HP:0000347;HP:0000175;HP:0000766;HP...,Face;Skeleton;Heart/Cardiovasculature/Lymphatic,18070134;16791849;15731757;16928994;16596670,DD,MSSE;ESS1,11772,2015-07-22 16:15:11,NaN,restricted repertoire of mutations,NaN,NaN,gain_of_function_variant,NaN


In [13]:
df_decipher[df_decipher['gene symbol']=='BMPR1A']

,gene symbol,gene mim,disease name,disease mim,confidence category,allelic requirement,mutation consequence,phenotypes,organ specificity list,pmids,panel,prev symbols,hgnc id,gene disease pair entry date,cross cutting modifier,mutation consequence flag,confidence value flag,comments,variant consequence,disease ontology


In [14]:
import pandas as pd


# Renombrar columnas para que coincidan
df_orphanet.rename(columns={'GeneSymbol': 'GeneSymbol', 'Name': 'DiseaseName'}, inplace=True)
df_decipher.rename(columns={'gene symbol': 'GeneSymbol', 'disease name': 'DiseaseName'}, inplace=True)

# Concatenar los DataFrames verticalmente
combined_df = pd.concat([df_orphanet, df_decipher], axis=0, ignore_index=True)

# Llenar NaN en la columna 'Disease' combinada con valores de 'DiseaseName' si existen
combined_df['Disease'] = combined_df['DiseaseName'].fillna('')

# Eliminar la columna 'DiseaseName' si ya no es necesaria
combined_df.drop(columns=['DiseaseName'], inplace=True)


/var/folders/17/bt4jbx0920sgdvwfqnkx2bnr0000gn/T/ipykernel_1379/3861784583.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_orphanet.rename(columns={'GeneSymbol': 'GeneSymbol', 'Name': 'DiseaseName'}, inplace=True)


In [15]:
combined_df.head(5)

,DisorderID,OrphaCode,ExpertLink,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus,gene mim,disease mim,...,prev symbols,hgnc id,gene disease pair entry date,cross cutting modifier,mutation consequence flag,confidence value flag,comments,variant consequence,disease ontology,Disease
0,205,337,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,activin A receptor type 1,ACVR1,2q24.1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fibrodysplasia ossificans progressiva
1,236,774,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,activin A receptor like type 1,ACVRL1,12q13.13,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hereditary hemorrhagic telangiectasia
2,17145,157794,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hereditary mixed polyposis syndrome
3,2010,2098,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Malformation syndrome,Disorder,bone morphogenetic protein receptor type 1B,BMPR1B,4q22.3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Acromesomelic dysplasia, Grebe type"
4,2413,2639,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Malformation syndrome,Disorder,bone morphogenetic protein receptor type 1B,BMPR1B,4q22.3,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fibular aplasia-complex brachydactyly syndrome


In [16]:
tgfbr1_disease=combined_df[combined_df['GeneSymbol']=='BMPR1A']
print(tgfbr1_disease['Disease'].unique())
tgfbr1_disease

['Hereditary mixed polyposis syndrome'
 'Generalized juvenile polyposis/juvenile polyposis coli'
 'Familial colorectal cancer Type X']


,DisorderID,OrphaCode,ExpertLink,DisorderType,DisorderGroup,GeneName,GeneSymbol,GeneLocus,gene mim,disease mim,...,prev symbols,hgnc id,gene disease pair entry date,cross cutting modifier,mutation consequence flag,confidence value flag,comments,variant consequence,disease ontology,Disease
2,17145,157794,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hereditary mixed polyposis syndrome
6,21951,329971,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Clinical subtype,Subtype of disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Generalized juvenile polyposis/juvenile polypo...
8,23409,440437,http://www.orpha.net/consor/cgi-bin/OC_Exp.php...,Disease,Disorder,bone morphogenetic protein receptor type 1A,BMPR1A,10q23.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Familial colorectal cancer Type X


In [17]:
df_clinvar = pd.read_csv('../data/mutaciones/clinvar_mutations')

In [18]:
df_clinvar.head(2)

,Protein change,Molecular consequence,VariationID,Condition(s),Variant type,Gene(s),Germline classification,Germline review status
0,R258M,missense variant,376364,Gastric adenocarcinoma,single nucleotide variant,ACVR1,Likely pathogenic,no assertion criteria provided
1,R258M,missense variant,376364,Brainstem glioma,single nucleotide variant,ACVR1,Likely pathogenic,no assertion criteria provided


In [19]:
df_clinvar[df_clinvar['Condition(s)']=='Familial colorectal cancer Type X']

,Protein change,Molecular consequence,VariationID,Condition(s),Variant type,Gene(s),Germline classification,Germline review status


In [20]:
# -merge de clinvar y combined_df por la columnas de genes
df_combined = pd.merge(df_clinvar, combined_df, left_on='Gene(s)', right_on='GeneSymbol')

# Filtrar las variaciones que están relacionadas con enfermedades raras
# filtrar por las columnas 'GeneSymbol', 'Disease', 'Protein change', 'Molecular consequence', 'VariationID','Germline classification'
rare_disease_variations = df_combined[['Gene(s)','GeneSymbol', 'Disease', 'Condition(s)','Protein change', 'Molecular consequence', 'VariationID','Germline classification']]


In [21]:
rare_disease_variations

,Gene(s),GeneSymbol,Disease,Condition(s),Protein change,Molecular consequence,VariationID,Germline classification
0,ACVR1,ACVR1,Fibrodysplasia ossificans progressiva,Gastric adenocarcinoma,R258M,missense variant,376364,Likely pathogenic
1,ACVR1,ACVR1,FIBRODYSPLASIA OSSIFICANS PROGRESSIVA,Gastric adenocarcinoma,R258M,missense variant,376364,Likely pathogenic
2,ACVR1,ACVR1,Fibrodysplasia ossificans progressiva,Brainstem glioma,R258M,missense variant,376364,Likely pathogenic
3,ACVR1,ACVR1,FIBRODYSPLASIA OSSIFICANS PROGRESSIVA,Brainstem glioma,R258M,missense variant,376364,Likely pathogenic
4,ACVR1,ACVR1,Fibrodysplasia ossificans progressiva,Brainstem glioma,G328V,missense variant,376363,Likely pathogenic
...,...,...,...,...,...,...,...,...
5255,TGFBR2,TGFBR2,Loeys-Dietz syndrome 2,Familial thoracic aortic aneurysm and aortic d...,E406Q,missense variant,12503,Pathogenic
5256,TGFBR2,TGFBR2,Loeys-Dietz syndrome 2,Familial thoracic aortic aneurysm and aortic d...,E527Q,missense variant,12503,Pathogenic
5257,TGFBR2,TGFBR2,Loeys-Dietz syndrome 2,Familial thoracic aortic aneurysm and aortic d...,E535Q,missense variant,12503,Pathogenic
5258,TGFBR2,TGFBR2,Loeys-Dietz syndrome 2,"Colorectal cancer, hereditary nonpolyposis, ty...",A535fs,frameshift variant,12501,Pathogenic


In [22]:
rare_disease_variations['Disease'].nunique()

18

In [23]:
rare_disease_variations['Protein change'].nunique()

2315

In [24]:
rare_disease_variations['Condition(s)'].unique()

array(['Gastric adenocarcinoma', 'Brainstem glioma',
       'Diffuse midline glioma, H3 K27M-mutant',
       'Progressive myositis ossificans', 'Inborn genetic diseases',
       'ACVR1-related disorder', 'Epicanthus',
       'Heterotaxy, visceral, 4, autosomal',
       'Hereditary hemorrhagic telangiectasia',
       'Telangiectasia, hereditary hemorrhagic, type 2',
       'Cardiovascular phenotype', 'ACVRL1-related disorder',
       'Abnormal bleeding', 'Thrombocytopenia',
       'Pulmonary arterial hypertension',
       'Telangiectasia, hereditary hemorrhagic, type 1', 'not specified',
       'Pulmonary arterial hypertension related to hereditary hemorrhagic telangiectasia',
       'Pulmonary hypertension, primary, 1',
       'Abnormality of the pulmonary vasculature', 'Epistaxis',
       'See cases', 'Juvenile polyposis syndrome',
       'Hereditary cancer-predisposing syndrome',
       'Polyposis syndrome, hereditary mixed, 2',
       'Generalized juvenile polyposis/juvenile polypos

In [25]:
# Guarda el DataFrame en un archivo Excel
rare_disease_variations.to_excel('rare_disease_variations2.xlsx', index=False)

In [26]:
rare_disease_variations[rare_disease_variations['GeneSymbol']=='ACVRL1']

,Gene(s),GeneSymbol,Disease,Condition(s),Protein change,Molecular consequence,VariationID,Germline classification
37,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Hereditary hemorrhagic telangiectasia,Y182fs,frameshift variant,3076067,Likely pathogenic
38,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Hereditary hemorrhagic telangiectasia,Y286fs,frameshift variant,3076067,Likely pathogenic
39,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Hereditary hemorrhagic telangiectasia,Y98fs,frameshift variant,3076067,Likely pathogenic
40,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Hereditary hemorrhagic telangiectasia,L135fs,frameshift variant|intron variant,3076019,Likely pathogenic
41,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Hereditary hemorrhagic telangiectasia,L267fs,frameshift variant,3068661,Likely pathogenic
...,...,...,...,...,...,...,...,...
742,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,"Telangiectasia, hereditary hemorrhagic, type 2",I398N,missense variant,8248,Pathogenic
743,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,"Telangiectasia, hereditary hemorrhagic, type 2",W50C,missense variant,8246,Pathogenic
744,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Cardiovascular phenotype,W50C,missense variant,8246,Pathogenic
745,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Cardiovascular phenotype,M376R,missense variant,8244,Pathogenic


In [27]:
rare_disease_variations.loc[rare_disease_variations['Condition(s)'].str.contains('hypertension')]

,Gene(s),GeneSymbol,Disease,Condition(s),Protein change,Molecular consequence,VariationID,Germline classification
515,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Pulmonary arterial hypertension,V216G,missense variant,812853,Likely pathogenic
629,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Pulmonary arterial hypertension related to her...,R484fs,frameshift variant,426038,Pathogenic
630,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,"Pulmonary hypertension, primary, 1",R484G,missense variant,426037,Pathogenic/Likely pathogenic
631,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,Pulmonary arterial hypertension,R484G,missense variant,426037,Pathogenic/Likely pathogenic
635,ACVRL1,ACVRL1,Hereditary hemorrhagic telangiectasia,"Pulmonary hypertension, primary, 1",A478D,missense variant,426034,Pathogenic
...,...,...,...,...,...,...,...,...
3050,BMPR2,BMPR2,Pulmonary venoocclusive disease,Pulmonary arterial hypertension,S73*,nonsense,8797,Pathogenic
3051,BMPR2,BMPR2,Drug- or toxin-induced pulmonary arterial hype...,Primary pulmonary hypertension,R899*,nonsense,8796,Pathogenic
3052,BMPR2,BMPR2,Pulmonary venoocclusive disease,Primary pulmonary hypertension,R899*,nonsense,8796,Pathogenic
3053,BMPR2,BMPR2,Drug- or toxin-induced pulmonary arterial hype...,"Pulmonary hypertension, primary, 1",R899*,nonsense,8796,Pathogenic


In [28]:
GEN=rare_disease_variations[rare_disease_variations['GeneSymbol']=='TGFBR2']

In [29]:
GEN['Protein change'].nunique()

845

In [30]:
import pandas as pd

# Crear un diccionario con los DataFrames por gen
genes = ['ACVR1', 'ACVR1B', 'ACVR1C', 'ACVR2A', 'ACVR2B', 'BMPR1A', 'BMPR1B', 'BMPR2', 'TGFBR1', 'TGFBR2', 'AMHR2', 'ACVRL1']
gene_dfs = {gene: rare_disease_variations[rare_disease_variations['GeneSymbol'] == gene] for gene in genes}


# Función para detallar mutaciones por gen
def detallar_mutaciones(df):
    return df.groupby(['GeneSymbol', 'Disease', 'Molecular consequence'])['Protein change'].apply(list).reset_index(name='mutaciones')

# Crear diccionarios para almacenar los resultados
detalles_por_gen = {}

# Aplicar las funciones a cada DataFrame en el diccionario
for gene, df in gene_dfs.items():
    detalles_por_gen[gene] = detallar_mutaciones(df)


In [31]:
print("Detalle de mutaciones:")
detalles_por_gen['BMPR1A']

Detalle de mutaciones:


,GeneSymbol,Disease,Molecular consequence,mutaciones
0,BMPR1A,Familial colorectal cancer Type X,frameshift variant,"[F147fs, I122fs, T391fs, Q117fs, G65fs, S113fs..."
1,BMPR1A,Familial colorectal cancer Type X,frameshift variant|5 prime UTR variant|non-cod...,[D27fs]
2,BMPR1A,Familial colorectal cancer Type X,frameshift variant|5 prime UTR variant|non-cod...,[R9fs]
3,BMPR1A,Familial colorectal cancer Type X,frameshift variant|non-coding transcript variant,"[Y192fs, Y278fs, Y304fs, Y306fs, Y322fs, Y331f..."
4,BMPR1A,Familial colorectal cancer Type X,frameshift variant|non-coding transcript varia...,"[S171fs, S143fs, V162fs, V134fs, V134fs, V162f..."
5,BMPR1A,Familial colorectal cancer Type X,inframe_deletion|initiator_codon_variant,[M1del]
6,BMPR1A,Familial colorectal cancer Type X,missense variant,"[C100Y, C72Y, G71R, G99R, H232D, H318D, H344D,..."
7,BMPR1A,Familial colorectal cancer Type X,missense variant|initiator_codon_variant,"[M1I, M1I, M1L, M1L, M1I, M1I, M1V, M1V]"
8,BMPR1A,Familial colorectal cancer Type X,nonsense,"[W274*, W221*, W247*, W265*, W135*, W249*, C10..."
9,BMPR1A,Familial colorectal cancer Type X,nonsense|5 prime UTR variant|non-coding transc...,[Y5*]


In [32]:
import pandas as pd

# Supongamos que 'rare_disease_variations' es tu DataFrame
# Cambiar el nombre de las filas en la columna 'Disease'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'Brachydactyly type A1', 'Disease'] = 'Brachydactyly type A1, A2, C'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'Brachydactyly type C', 'Disease'] = 'Brachydactyly type A1, A2, C'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'Brachydactyly type A2', 'Disease'] = 'Brachydactyly type A1, A2, C'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'BRACHYDACTYLY TYPE A2', 'Disease'] = 'Brachydactyly type A1, A2, C'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'LOEYS-DIETZ SYNDROME', 'Disease'] = 'Loeys-Dietz syndrome'
rare_disease_variations.loc[rare_disease_variations['Disease'] == 'FIBRODYSPLASIA OSSIFICANS PROGRESSIVA', 'Disease'] = 'Fibrodysplasia ossificans progressiva'


In [33]:
# crea una lista con los 18 tipos de cancer a tratar
lista_rd = rare_disease_variations['Disease'].unique()
#Crea un dataframe para cada tipo de grafico, para poder graficarlos de forma independiente
dict_rd = {}

# Iterar sobre cada tipo de cáncer en lista_cancer
for tipo in lista_rd:
    # Filtrar el DataFrame original para obtener solo las filas correspondientes al tipo de cáncer actual
    df_disease = rare_disease_variations[rare_disease_variations['Disease'] == tipo]
    
    # Asignar el DataFrame filtrado al diccionario con la clave correspondiente al tipo de cáncer
    dict_rd[tipo] = df_disease

# Ahora se puede acceder a cada DataFrame utilizando la clave correspondiente al tipo de cáncer en el diccionario



In [34]:
for Disease, data in sorted(dict_rd.items(), key=lambda x: len(x[1]['Protein change'].unique()), reverse=True):
    print(f"Número de mutaciones en enfermedades raras {Disease}: {len(data['Protein change'].unique())}")


Número de mutaciones en enfermedades raras Loeys-Dietz syndrome 2: 845
Número de mutaciones en enfermedades raras Hereditary hemorrhagic telangiectasia: 534
Número de mutaciones en enfermedades raras Hereditary mixed polyposis syndrome: 353
Número de mutaciones en enfermedades raras Generalized juvenile polyposis/juvenile polyposis coli: 353
Número de mutaciones en enfermedades raras Familial colorectal cancer Type X: 353
Número de mutaciones en enfermedades raras Drug- or toxin-induced pulmonary arterial hypertension: 330
Número de mutaciones en enfermedades raras Pulmonary venoocclusive disease: 330
Número de mutaciones en enfermedades raras Marfan syndrome type 2: 258
Número de mutaciones en enfermedades raras Multiple self-healing squamous epithelioma: 258
Número de mutaciones en enfermedades raras Loeys-Dietz syndrome: 258
Número de mutaciones en enfermedades raras Acromesomelic dysplasia, Grebe type: 22
Número de mutaciones en enfermedades raras Fibular aplasia-complex brachydact

In [35]:
for Disease, data in sorted(dict_rd.items(), key=lambda x: len(x[1]['Protein change'].unique()), reverse=True):
    print(f"Número de mutaciones en enfermedades raras {Disease}: {len(data['Protein change'].unique())}")


Número de mutaciones en enfermedades raras Loeys-Dietz syndrome 2: 845
Número de mutaciones en enfermedades raras Hereditary hemorrhagic telangiectasia: 534
Número de mutaciones en enfermedades raras Hereditary mixed polyposis syndrome: 353
Número de mutaciones en enfermedades raras Generalized juvenile polyposis/juvenile polyposis coli: 353
Número de mutaciones en enfermedades raras Familial colorectal cancer Type X: 353
Número de mutaciones en enfermedades raras Drug- or toxin-induced pulmonary arterial hypertension: 330
Número de mutaciones en enfermedades raras Pulmonary venoocclusive disease: 330
Número de mutaciones en enfermedades raras Marfan syndrome type 2: 258
Número de mutaciones en enfermedades raras Multiple self-healing squamous epithelioma: 258
Número de mutaciones en enfermedades raras Loeys-Dietz syndrome: 258
Número de mutaciones en enfermedades raras Acromesomelic dysplasia, Grebe type: 22
Número de mutaciones en enfermedades raras Fibular aplasia-complex brachydact

In [36]:
import altair as alt
alt.data_transformers.enable('default', max_rows=None)

g1=alt.Chart(rare_disease_variations).mark_bar().encode(
    x=alt.X('Disease',axis=alt.Axis(
              labelAngle=40, labelFontSize=12), sort='-y'),
    y=alt.Y('count(Protein change):Q').title('Numero de mutaciones'),
    color=alt.Color('GeneSymbol', scale=alt.Scale(scheme='category20'), title='Genes')
).properties(
    title='Numero de mutaciones en enfermedades raras'
)

g2=alt.Chart(rare_disease_variations).mark_arc().encode(
    theta='count(Protein change)):Q',
    color="Molecular consequence"
).properties(
    title='Tipo de mutacion mas frecuente en cancer'
)
g1

alt.Chart(...)

In [37]:
alt.Chart(rare_disease_variations).transform_aggregate(
    unique_aa_mutation='distinct(Protein change)',
    groupby=['GeneSymbol','Disease', 'Protein change']
).mark_bar().encode(
    y=alt.Y('count(unique_aa_mutation):Q', title='Número de mutaciones'),
    x=alt.X('Disease', sort='-y', title='Enfermedad rara',axis=alt.Axis(
              labelAngle=-40, labelFontSize=10)),
    color=alt.Color('GeneSymbol', scale=alt.Scale(scheme='category20'), title='Genes')
).properties(
    title="Número de mutaciones en enfermedades raras"
).properties(
        width=600,
        height=500
    )

alt.Chart(...)

In [38]:
df_variaciones = rare_disease_variations[['GeneSymbol','Disease', 'Protein change']]

# Paso 1: Eliminar duplicados por paciente y gen, pero considerando solo mutaciones únicas por gen y paciente
df_variaciones = df_variaciones.drop_duplicates(subset=['Protein change','GeneSymbol','Disease'])
# Paso 2: Contar la frecuencia de mutación por gen (mutaciones únicas por paciente)
mutation_frequencies = df_variaciones.groupby(['Disease','GeneSymbol']).size().reset_index(name='mutation_count')

# Paso 3: Contar el número total de pacientes
total_mutation = rare_disease_variations['Protein change'].nunique()

# Paso 4: Ordenar de mayor a menor
mutation_frequencies.sort_values(ascending=False, by='mutation_count', inplace=True)

# Paso 5: Agrega la frecuencia acumulada
mutation_frequencies["cum_frequency"] = mutation_frequencies["mutation_count"].cumsum()
# Paso 6: Contar el numero total de mutaciones
sum_count = mutation_frequencies['mutation_count'].sum()
# Paso 7: Calcula la frecuencia acumulada porcentual
mutation_frequencies['%cum_frequency'] = mutation_frequencies['cum_frequency']/sum_count
# Paso 8: ordena los resultados
mutation_frequencies.sort_values(ascending=False, by='mutation_count', inplace=True)
mutation_frequencies


,Disease,GeneSymbol,mutation_count,cum_frequency,%cum_frequency
11,Loeys-Dietz syndrome 2,TGFBR2,845,845,0.213870
8,Hereditary hemorrhagic telangiectasia,ACVRL1,534,1379,0.349026
3,Familial colorectal cancer Type X,BMPR1A,353,1732,0.438370
6,Generalized juvenile polyposis/juvenile polypo...,BMPR1A,353,2085,0.527715
9,Hereditary mixed polyposis syndrome,BMPR1A,353,2438,0.617059
2,Drug- or toxin-induced pulmonary arterial hype...,BMPR2,330,2768,0.700582
14,Pulmonary venoocclusive disease,BMPR2,330,3098,0.784105
10,Loeys-Dietz syndrome,TGFBR1,258,3356,0.849405
12,Marfan syndrome type 2,TGFBR1,258,3614,0.914705
13,Multiple self-healing squamous epithelioma,TGFBR1,258,3872,0.980005


In [39]:
# Paso 1: Eliminar duplicados por paciente y gen, pero considerando solo mutaciones únicas por gen y paciente
df_variaciones = df_variaciones.drop_duplicates(subset=['Protein change','GeneSymbol','Disease'])
# Paso 2: Contar la frecuencia de mutación por gen (mutaciones únicas por paciente)
mutation_frequencies_2 = df_variaciones.groupby(['Disease','GeneSymbol']).size().reset_index(name='mutation_count')
mutation_frequencies_2.sort_values(by='mutation_count', ascending=False, inplace=True)
mutation_frequencies_2.reset_index(inplace = True, drop=True)
frecuencia_relativa = mutation_frequencies_2["mutation_count"].cumsum()
total_defectos = mutation_frequencies_2["mutation_count"].sum()
mutation_frequencies_2["relativa_acum_porcentual"] = frecuencia_relativa/total_defectos*100
mutation_frequencies_2

,Disease,GeneSymbol,mutation_count,relativa_acum_porcentual
0,Loeys-Dietz syndrome 2,TGFBR2,845,21.386991
1,Hereditary hemorrhagic telangiectasia,ACVRL1,534,34.902556
2,Familial colorectal cancer Type X,BMPR1A,353,43.837003
3,Generalized juvenile polyposis/juvenile polypo...,BMPR1A,353,52.771450
4,Hereditary mixed polyposis syndrome,BMPR1A,353,61.705897
5,Drug- or toxin-induced pulmonary arterial hype...,BMPR2,330,70.058213
6,Pulmonary venoocclusive disease,BMPR2,330,78.410529
7,Loeys-Dietz syndrome,TGFBR1,258,84.940521
8,Marfan syndrome type 2,TGFBR1,258,91.470514
9,Multiple self-healing squamous epithelioma,TGFBR1,258,98.000506


In [40]:
fig, ax = plt.subplots()

ax.bar(mutation_frequencies_2['Disease'], mutation_frequencies_2["mutation_count"], color="C0")
# Ajustar las etiquetas del eje x
ax.set_xticks(range(len(mutation_frequencies_2['Disease'])))  # Colocar las etiquetas en las posiciones correctas
ax.set_xticklabels(mutation_frequencies_2['Disease'], rotation=45, ha="right")  # Rotar las etiquetas y alinearlas a la derecha
ax2 = ax.twinx()
ax2.plot(df.index, mutation_frequencies_2["relativa_acum_porcentual"], color="C1", marker="o", ms=5)
# Ajustar el diseño para evitar solapamientos
plt.tight_layout()
plt.show() 

NameError: name 'plt' is not defined

In [41]:
fig, ax = plt.subplots()

# Gráfico de barras para el número de mutaciones
bars = ax.bar(mutation_frequencies_2['Disease'], mutation_frequencies_2["mutation_count"], color="C0")

# Crear el segundo eje y para la frecuencia acumulada
ax2 = ax.twinx()

# Gráfico de línea para la frecuencia acumulada
line = ax2.plot(mutation_frequencies_2['Disease'], mutation_frequencies_2["relativa_acum_porcentual"], color="C1", marker="o", ms=5)

# Agregar etiquetas con el número de mutaciones encima de las barras
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height, f'{int(height)}', ha='center', va='bottom', color='black', fontsize=9)

# Agregar etiquetas con el porcentaje acumulado en los puntos de la línea
for i, value in enumerate(mutation_frequencies_2["relativa_acum_porcentual"]):
    ax2.text(i, value, f'{value:.0f}%', ha='center', va='bottom', color='red', fontsize=9)

# Ajustar las etiquetas del eje x
ax.set_xticks(range(len(mutation_frequencies_2['Disease'])))  # Colocar las etiquetas en las posiciones correctas
ax.set_xticklabels(mutation_frequencies_2['Disease'], rotation=45, ha="right")  # Rotar las etiquetas y alinearlas a la derecha


# Mostrar el gráfico
plt.show()
 

NameError: name 'plt' is not defined

In [42]:
import altair as alt
import pandas as pd

# Gráfico de barras
base = alt.Chart(mutation_frequencies
).mark_bar().encode(
    y=alt.Y('mutation_count:Q', title='Número de mutaciones'),
    x=alt.X('Disease:N', sort='-y', title='Enfermedad rara', axis=alt.Axis(
              labelAngle=-40, labelFontSize=10)),
    color=alt.Color('GeneSymbol:N', scale=alt.Scale(scheme='category20'), title='Genes')
).properties(
    title="Número de mutaciones en enfermedades raras",
    width=600,
    height=500
)

# Línea de frecuencia acumulada
line = alt.Chart(mutation_frequencies
).mark_line(color='red').encode(
    y=alt.Y('%cum_frequency:Q', title='Frecuencia acumulada (%)', ,  scale=alt.Scale(domain=[0, 1])),
    x=alt.X('Disease:N', sort='-y'),
)

# Combinar ambos gráficos, con diferentes escalas en el eje y
combined = alt.layer(
    base,
    line
).resolve_scale(
    y='independent'  # Escalas independientes para el gráfico de barras y la línea
).properties(width=650, height=450)

combined


SyntaxError: invalid syntax (2086441853.py, line 20)

In [43]:
import altair as alt

# Datos
mutation_frequencies = pd.DataFrame({
    'Disease': ['0','Loeys-Dietz syndrome 2', 'Hereditary hemorrhagic telangiectasia', 'Familial colorectal cancer Type X', 
                'Generalized juvenile polyposis', 'Hereditary mixed polyposis syndrome', 
                'Drug- or toxin-induced pulmonary arterial hypertension', 'Pulmonary venoocclusive disease', 
                'Loeys-Dietz syndrome', 'Marfan syndrome type 2', 'Multiple self-healing squamous epithelioma',
                'Acromesomelic dysplasia, Grebe type', 'Brachydactyly type A1, A2, C', 
                'Fibular aplasia-complex brachydactyly syndrome', 'Fibrodysplasia ossificans progressiva', 'HETEROTAXY SYNDROME'],
    'GeneSymbol': ['0','TGFBR2', 'ACVRL1', 'BMPR1A', 'BMPR1A', 'BMPR1A', 'BMPR2', 'BMPR2', 'TGFBR1', 'TGFBR1', 
                   'TGFBR1', 'BMPR1B', 'BMPR1B', 'BMPR1B', 'ACVR1', 'ACVR2B'],
    'mutation_count': [0,845, 534, 353, 353, 353, 330, 330, 258, 258, 258, 22, 22, 22, 12, 1],
    'cum_frequency': [0,845, 1379, 1732, 2085, 2438, 2768, 3098, 3356, 3614, 3872, 3894, 3916, 3938, 3950, 3951],
    '%cum_frequency': [0,0.213870, 0.349026, 0.438370, 0.527715, 0.617059, 0.700582, 0.784105, 0.849405, 0.914705, 
                       0.980005, 0.985573, 0.991141, 0.996710, 0.999747, 1.000000]
})

# Gráfico de barras para número de mutaciones
bars = alt.Chart(mutation_frequencies).mark_bar().encode(
    y=alt.Y('%cum_frequency:Q', axis=alt.Axis(title='Frecuencia acumulada (%)')),
    x=alt.X('Disease:N', sort='y', title='Enfermedad rara', axis=alt.Axis(labelAngle=-45)),
    color=alt.Color('GeneSymbol:N', scale=alt.Scale(scheme='category20'), title='Genes')
).properties(
    width=600,
    height=400
)

# Línea de frecuencia acumulada
line = alt.Chart(mutation_frequencies).mark_line(color='red').encode(
    y=alt.Y('mutation_count:Q', title='Número de mutaciones'),
    x=alt.X('Disease:N', sort='-y')
)

# Combina ambos gráficos, con diferentes escalas en el eje y
combined_chart = alt.layer(bars, line).resolve_scale(
    y='independent'  # Escalas independientes para ambos gráficos
)

combined_chart



alt.LayerChart(...)

In [44]:
# Cambiar el nombre de las filas en la columna 'Molecular consequence'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'frameshift variant|non-coding transcript variant', 'Molecular consequence'] = 'frameshift variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'missense variant|intron variant', 'Molecular consequence'] = 'missense variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'nonsense|non-coding transcript variant', 'Molecular consequence'] = 'nonsense'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'frameshift variant|non-coding transcript variant|intron variant', 'Molecular consequence'] = 'frameshift variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'missense variant|non-coding transcript variant', 'Molecular consequence'] = 'missense variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'nonsense|non-coding transcript variant|intron variant', 'Molecular consequence'] = 'nonsense'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'missense variant|non-coding transcript variant|intron variant', 'Molecular consequence'] = 'missense variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'nonsense|intron variant', 'Molecular consequence'] = 'nonsense'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'frameshift variant|intron variant', 'Molecular consequence'] = 'frameshift variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'missense variant|initiator_codon_variant', 'Molecular consequence'] = 'missense variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'missense variant|splice donor variant', 'Molecular consequence'] = 'missense variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'splice donor variant|missense variant', 'Molecular consequence'] = 'missense variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'frameshift variant|initiator_codon_variant|5 prime UTR variant|non-coding transcript variant|intron variant', 'Molecular consequence'] = 'frameshift variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'frameshift variant|5 prime UTR variant|non-coding transcript variant', 'Molecular consequence'] = 'frameshift variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'nonsense|5 prime UTR variant|non-coding transcript variant|intron variant', 'Molecular consequence'] = 'nonsense'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'intron variant|missense variant', 'Molecular consequence'] = 'missense variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'intron variant|frameshift variant', 'Molecular consequence'] = 'frameshift variant'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'inframe_deletion|initiator_codon_variant', 'Molecular consequence'] = 'inframe_deletion'
rare_disease_variations.loc[rare_disease_variations['Molecular consequence'] == 'frameshift variant|5 prime UTR variant|non-coding transcript variant|intron variant', 'Molecular consequence'] = 'frameshift variant'


In [45]:
df_consequence = rare_disease_variations[['Protein change','Molecular consequence']]

# Paso 1: Eliminar duplicados por paciente y gen, pero considerando solo mutaciones únicas por gen y paciente
df_consequence = df_consequence.drop_duplicates(subset=['Protein change','Molecular consequence'])

# Paso 2: Contar la frecuencia de mutación por gen (mutaciones únicas por paciente)
mutation_frequencies_consequence = df_consequence.groupby(['Molecular consequence']).size().reset_index(name='mutation_count')

# Paso 3: Contar el número total de pacientes
total_mutation = rare_disease_variations['Protein change'].nunique()

# Paso 4: Calcular la frecuencia de mutación como la proporción de mutaciones por gen al número total de pacientes
mutation_frequencies_consequence['mutation_frequency'] = (mutation_frequencies_consequence['mutation_count'] / total_mutation)*100

# Paso 5: Mostrar los resultados
mutation_frequencies_consequence.sort_values(ascending=False, by='mutation_count')

,Molecular consequence,mutation_count,mutation_frequency
2,missense variant,1363,58.876890
0,frameshift variant,548,23.671706
3,nonsense,386,16.673866
1,inframe_deletion,18,0.777538


In [46]:
molecular_consequence = mutation_frequencies_consequence.loc[2, 'Molecular consequence']
print( molecular_consequence)


missense variant


In [47]:
base = alt.Chart(mutation_frequencies_consequence).mark_arc().encode(
    theta=alt.Theta('mutation_count:Q', sort='ascending'),  # Ordenar de mayor a menor,
    color="Molecular consequence"
).properties(
    title='Tipo de mutacion mas frecuente en cancer'
)


base

alt.Chart(...)

In [48]:
# Agrupar por consequence_type y contar el número de mutaciones únicas
mutation_counts = rare_disease_variations.groupby('Molecular consequence')['Protein change'].nunique().reset_index(name='num_mutaciones')

# Ordenar por el número de mutaciones de forma descendente
mutation_counts = mutation_counts.sort_values(by='num_mutaciones', ascending=False)

# Imprimir el número de mutaciones para cada consequence_type
for index, row in mutation_counts.iterrows():
    print(f"Número de mutaciones {row['Molecular consequence']}: {row['num_mutaciones']}")

# Crear el histograma
base = alt.Chart(rare_disease_variations).transform_aggregate(
    unique_aa_mutation='distinct(Protein change)',
    groupby=['Molecular consequence','Protein change']
         ).mark_bar(
        ).encode(
            x=alt.X('Molecular consequence', title='Tipo de consecuencia', sort='-y',axis=alt.Axis(
              labelAngle=40, labelFontSize=12)),
            y=alt.Y('count(unique_aa_mutation):Q', title='Número de mutaciones'),
            color=alt.Color(scale=alt.Scale(scheme='category20')),
            tooltip=['Molecular consequence', 'count()'],
            text =  'count(unique_aa_mutation):Q'
        ).properties(
            title='Mutaciones consecuencia',
            width=300,
        height=150
            )
base.mark_bar() + base.mark_text(align='center', dx=2)

Número de mutaciones missense variant: 1363
Número de mutaciones frameshift variant: 548
Número de mutaciones nonsense: 386
Número de mutaciones inframe_deletion: 18


alt.LayerChart(...)

In [49]:
rare_disease_variations

,Gene(s),GeneSymbol,Disease,Condition(s),Protein change,Molecular consequence,VariationID,Germline classification
0,ACVR1,ACVR1,Fibrodysplasia ossificans progressiva,Gastric adenocarcinoma,R258M,missense variant,376364,Likely pathogenic
1,ACVR1,ACVR1,Fibrodysplasia ossificans progressiva,Gastric adenocarcinoma,R258M,missense variant,376364,Likely pathogenic
2,ACVR1,ACVR1,Fibrodysplasia ossificans progressiva,Brainstem glioma,R258M,missense variant,376364,Likely pathogenic
3,ACVR1,ACVR1,Fibrodysplasia ossificans progressiva,Brainstem glioma,R258M,missense variant,376364,Likely pathogenic
4,ACVR1,ACVR1,Fibrodysplasia ossificans progressiva,Brainstem glioma,G328V,missense variant,376363,Likely pathogenic
...,...,...,...,...,...,...,...,...
5255,TGFBR2,TGFBR2,Loeys-Dietz syndrome 2,Familial thoracic aortic aneurysm and aortic d...,E406Q,missense variant,12503,Pathogenic
5256,TGFBR2,TGFBR2,Loeys-Dietz syndrome 2,Familial thoracic aortic aneurysm and aortic d...,E527Q,missense variant,12503,Pathogenic
5257,TGFBR2,TGFBR2,Loeys-Dietz syndrome 2,Familial thoracic aortic aneurysm and aortic d...,E535Q,missense variant,12503,Pathogenic
5258,TGFBR2,TGFBR2,Loeys-Dietz syndrome 2,"Colorectal cancer, hereditary nonpolyposis, ty...",A535fs,frameshift variant,12501,Pathogenic
